# Customer Engagement & Product Utilization Analytics for Retention Strategy

## Notebook 02: Data Cleaning & Validation

### Notebook Objective

The objective of this notebook is to validate the raw banking dataset and prepare a clean dataset for deeper analysis and feature engineering.

Although the initial inspection showed that the dataset does not contain missing values or duplicate records, this notebook performs additional validation checks to ensure that the data is reliable, consistent, and suitable for customer engagement and retention analysis.

---

### Key Validation Areas

In this notebook, we will validate:

- Column naming consistency
- Data types
- Missing values
- Duplicate records
- Customer ID uniqueness
- Binary columns
- Categorical columns
- Numeric ranges
- Zero-balance customers
- Product count distribution
- Churn target consistency
- Business logic checks

---

### Final Output

At the end of this notebook, we will export a cleaned dataset to:

`data/cleaned/european_bank_cleaned.csv`

### Import Libraries

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
print("Libraries are imported successfully")

Libraries are imported successfully


### Define Paths

In [3]:
# Define project paths
BASE_DIR = Path.cwd().parent

RAW_DATA_PATH = BASE_DIR / "data" / "raw" / "European_Bank.csv"
CLEANED_DATA_DIR = BASE_DIR / "data" / "cleaned"
CLEANED_DATA_PATH = CLEANED_DATA_DIR / "european_bank_cleaned.csv"

print("Raw Data Path:", RAW_DATA_PATH)
print("Cleaned Data Path:", CLEANED_DATA_PATH)

Raw Data Path: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\raw\European_Bank.csv
Cleaned Data Path: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\cleaned\european_bank_cleaned.csv


### Load Raw Dataset

In [4]:
# Load raw dataset
df = pd.read_csv(RAW_DATA_PATH)

# Preview dataset
df.head()

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,"101,348.88",1
1,2025,15647311,Hill,608,Spain,Female,41,1,"83,807.86",1,0,1,"112,542.58",0
2,2025,15619304,Onio,502,France,Female,42,8,"159,660.80",3,1,0,"113,931.57",1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,"93,826.63",0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,"125,510.82",1,1,1,"79,084.10",0


### Create Working Copy

In [5]:
# Create a working copy for cleaning and validation
bank_df = df.copy()

print(f"Working dataset created successfully with {bank_df.shape[0]:,} rows and {bank_df.shape[1]:,} columns.")

Working dataset created successfully with 10,000 rows and 14 columns.


### Column Name Validation

In [6]:
# Check current column names
bank_df.columns.tolist()

['Year',
 'CustomerId',
 'Surname',
 'CreditScore',
 'Geography',
 'Gender',
 'Age',
 'Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary',
 'Exited']

### Check for Spaces in Column Names

In [7]:
# Check if any column names contain leading/trailing spaces
column_space_check = pd.DataFrame({
    "Original Column": bank_df.columns,
    "Cleaned Column": bank_df.columns.str.strip(),
    "Has Leading/Trailing Space": bank_df.columns != bank_df.columns.str.strip()
})

column_space_check

,Original Column,Cleaned Column,Has Leading/Trailing Space
0,Year,Year,False
1,CustomerId,CustomerId,False
2,Surname,Surname,False
3,CreditScore,CreditScore,False
4,Geography,Geography,False
5,Gender,Gender,False
6,Age,Age,False
7,Tenure,Tenure,False
8,Balance,Balance,False
9,NumOfProducts,NumOfProducts,False


### Standardize Column Names

In [8]:
# Standardize column names for consistency
bank_df.columns = bank_df.columns.str.strip()

# Confirm updated column names
bank_df.columns.tolist()

['Year',
 'CustomerId',
 'Surname',
 'CreditScore',
 'Geography',
 'Gender',
 'Age',
 'Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary',
 'Exited']

### Data Type Validation

In [9]:
# Check data types
data_types = bank_df.dtypes.reset_index()
data_types.columns = ["Column", "Data Type"]

data_types

,Column,Data Type
0,Year,int64
1,CustomerId,int64
2,Surname,object
3,CreditScore,int64
4,Geography,object
5,Gender,object
6,Age,int64
7,Tenure,int64
8,Balance,float64
9,NumOfProducts,int64


### Data Type Summary

In [10]:
# Count columns by data type
bank_df.dtypes.value_counts()

int64      9
object     3
float64    2
Name: count, dtype: int64

### Missing Values Recheck

In [11]:
# Recheck missing values
missing_summary = bank_df.isnull().sum().reset_index()
missing_summary.columns = ["Column", "Missing Values"]
missing_summary["Missing %"] = (missing_summary["Missing Values"] / len(bank_df) * 100).round(2)

missing_summary

,Column,Missing Values,Missing %
0,Year,0,0.00
1,CustomerId,0,0.00
2,Surname,0,0.00
3,CreditScore,0,0.00
4,Geography,0,0.00
5,Gender,0,0.00
6,Age,0,0.00
7,Tenure,0,0.00
8,Balance,0,0.00
9,NumOfProducts,0,0.00


### Duplicate Records Recheck

In [12]:
# Recheck duplicate rows
duplicate_count = bank_df.duplicated().sum()

print(f"Total Duplicate Rows: {duplicate_count}")

Total Duplicate Rows: 0


### Customer ID Uniqueness Recheck

In [13]:
# Validate CustomerId uniqueness
customer_id_summary = {
    "Total Records": bank_df["CustomerId"].count(),
    "Unique Customer IDs": bank_df["CustomerId"].nunique(),
    "Duplicate Customer IDs": bank_df["CustomerId"].count() - bank_df["CustomerId"].nunique()
}

customer_id_summary

{'Total Records': np.int64(10000),
 'Unique Customer IDs': 10000,
 'Duplicate Customer IDs': np.int64(0)}

### Binary Columns Validation

In [15]:
# Validate binary columns
binary_columns = ["HasCrCard", "IsActiveMember", "Exited"]

binary_validation = {}

for col in binary_columns:
    binary_validation[col] = sorted(bank_df[col].astype(int).unique().tolist())

binary_validation

{'HasCrCard': [0, 1], 'IsActiveMember': [0, 1], 'Exited': [0, 1]}

### Categorical Columns Validation

In [16]:
# Validate categorical columns
categorical_columns = ["Geography", "Gender"]

for col in categorical_columns:
    print(f"\n{col} Distribution:")
    print(bank_df[col].value_counts())


Geography Distribution:
Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

Gender Distribution:
Gender
Male      5457
Female    4543
Name: count, dtype: int64


### Numeric Range Validation

In [17]:
# Validate numeric column ranges
numeric_columns = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "EstimatedSalary"
]

numeric_range_validation = bank_df[numeric_columns].agg(["min", "max", "mean", "median"]).T

numeric_range_validation

,min,max,mean,median
CreditScore,350.00,850.00,650.53,652.00
Age,18.00,92.00,38.92,37.00
Tenure,0.00,10.00,5.01,5.00
Balance,0.00,"250,898.09","76,485.89","97,198.54"
NumOfProducts,1.00,4.00,1.53,1.00
EstimatedSalary,11.58,"199,992.48","100,090.24","100,193.91"


### Invalid Value Checks

In [18]:
# Check for invalid or impossible values
invalid_checks = {
    "Negative CreditScore": (bank_df["CreditScore"] < 0).sum(),
    "Negative Age": (bank_df["Age"] < 0).sum(),
    "Negative Tenure": (bank_df["Tenure"] < 0).sum(),
    "Negative Balance": (bank_df["Balance"] < 0).sum(),
    "Negative NumOfProducts": (bank_df["NumOfProducts"] < 0).sum(),
    "Negative EstimatedSalary": (bank_df["EstimatedSalary"] < 0).sum(),
    "Invalid Product Count": (~bank_df["NumOfProducts"].between(1, 4)).sum(),
    "Invalid HasCrCard Values": (~bank_df["HasCrCard"].isin([0, 1])).sum(),
    "Invalid IsActiveMember Values": (~bank_df["IsActiveMember"].isin([0, 1])).sum(),
    "Invalid Exited Values": (~bank_df["Exited"].isin([0, 1])).sum()
}

pd.DataFrame.from_dict(invalid_checks, orient="index", columns=["Invalid Count"])

,Invalid Count
Negative CreditScore,0
Negative Age,0
Negative Tenure,0
Negative Balance,0
Negative NumOfProducts,0
Negative EstimatedSalary,0
Invalid Product Count,0
Invalid HasCrCard Values,0
Invalid IsActiveMember Values,0
Invalid Exited Values,0


### Year Column Validation

In [19]:
# Validate year column
year_distribution = bank_df["Year"].value_counts().sort_index().reset_index()
year_distribution.columns = ["Year", "Customer Count"]

year_distribution

,Year,Customer Count
0,2025,10000


### Product Count Distribution

In [20]:
# Product count distribution
product_distribution = bank_df["NumOfProducts"].value_counts().sort_index().reset_index()
product_distribution.columns = ["Number of Products", "Customer Count"]

product_distribution["Customer %"] = (
    product_distribution["Customer Count"] / len(bank_df) * 100
).round(2)

product_distribution

,Number of Products,Customer Count,Customer %
0,1,5084,50.84
1,2,4590,45.90
2,3,266,2.66
3,4,60,0.60


### Zero Balance Customers Check

In [21]:
# Check zero balance customers
zero_balance_customers = (bank_df["Balance"] == 0).sum()
non_zero_balance_customers = (bank_df["Balance"] > 0).sum()

zero_balance_summary = pd.DataFrame({
    "Balance Segment": ["Zero Balance", "Non-Zero Balance"],
    "Customer Count": [zero_balance_customers, non_zero_balance_customers]
})

zero_balance_summary["Customer %"] = (
    zero_balance_summary["Customer Count"] / len(bank_df) * 100
).round(2)

zero_balance_summary

,Balance Segment,Customer Count,Customer %
0,Zero Balance,3617,36.17
1,Non-Zero Balance,6383,63.83


### Zero Balance vs Churn Check

In [23]:
# Compare churn behavior for zero vs non-zero balance customers
balance_churn_check = bank_df.copy()

balance_churn_check["Balance Segment"] = np.where(
    balance_churn_check["Balance"] == 0,
    "Zero Balance",
    "Non-Zero Balance"
)

balance_churn_summary = (
    balance_churn_check
    .groupby("Balance Segment")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean")
    )
    .reset_index()
)

balance_churn_summary["Churn_Rate"] = (balance_churn_summary["Churn_Rate"] * 100).round(2)

balance_churn_summary

,Balance Segment,Customer_Count,Churned_Customers,Churn_Rate
0,Non-Zero Balance,6383,1537,24.08
1,Zero Balance,3617,500,13.82


### Engagement Distribution

In [24]:
# Engagement distribution based on IsActiveMember
engagement_distribution = bank_df["IsActiveMember"].value_counts().reset_index()
engagement_distribution.columns = ["IsActiveMember", "Customer Count"]

engagement_distribution["Engagement Status"] = engagement_distribution["IsActiveMember"].map({
    0: "Inactive Member",
    1: "Active Member"
})

engagement_distribution["Customer %"] = (
    engagement_distribution["Customer Count"] / len(bank_df) * 100
).round(2)

engagement_distribution = engagement_distribution[
    ["Engagement Status", "IsActiveMember", "Customer Count", "Customer %"]
]

engagement_distribution

,Engagement Status,IsActiveMember,Customer Count,Customer %
0,Active Member,1,5151,51.51
1,Inactive Member,0,4849,48.49


### Engagement vs Churn Validation

In [25]:
# Analyze churn rate by engagement status
engagement_churn_summary = (
    bank_df
    .groupby("IsActiveMember")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean")
    )
    .reset_index()
)

engagement_churn_summary["Engagement Status"] = engagement_churn_summary["IsActiveMember"].map({
    0: "Inactive Member",
    1: "Active Member"
})

engagement_churn_summary["Churn_Rate"] = (
    engagement_churn_summary["Churn_Rate"] * 100
).round(2)

engagement_churn_summary = engagement_churn_summary[
    ["Engagement Status", "Customer_Count", "Churned_Customers", "Churn_Rate"]
]

engagement_churn_summary

,Engagement Status,Customer_Count,Churned_Customers,Churn_Rate
0,Inactive Member,4849,1302,26.85
1,Active Member,5151,735,14.27


### Product Count vs Churn Validation

In [26]:
# Analyze churn rate by number of products
product_churn_summary = (
    bank_df
    .groupby("NumOfProducts")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean")
    )
    .reset_index()
)

product_churn_summary["Churn_Rate"] = (
    product_churn_summary["Churn_Rate"] * 100
).round(2)

product_churn_summary

,NumOfProducts,Customer_Count,Churned_Customers,Churn_Rate
0,1,5084,1409,27.71
1,2,4590,348,7.58
2,3,266,220,82.71
3,4,60,60,100.00


### Credit Card Ownership vs Churn Validation

In [27]:
# Analyze churn rate by credit card ownership
credit_card_churn_summary = (
    bank_df
    .groupby("HasCrCard")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean")
    )
    .reset_index()
)

credit_card_churn_summary["Credit Card Status"] = credit_card_churn_summary["HasCrCard"].map({
    0: "No Credit Card",
    1: "Has Credit Card"
})

credit_card_churn_summary["Churn_Rate"] = (
    credit_card_churn_summary["Churn_Rate"] * 100
).round(2)

credit_card_churn_summary = credit_card_churn_summary[
    ["Credit Card Status", "Customer_Count", "Churned_Customers", "Churn_Rate"]
]

credit_card_churn_summary

,Credit Card Status,Customer_Count,Churned_Customers,Churn_Rate
0,No Credit Card,2945,613,20.81
1,Has Credit Card,7055,1424,20.18


### Geography vs Churn Validation

In [28]:
# Analyze churn rate by geography
geography_churn_summary = (
    bank_df
    .groupby("Geography")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean")
    )
    .reset_index()
)

geography_churn_summary["Churn_Rate"] = (
    geography_churn_summary["Churn_Rate"] * 100
).round(2)

geography_churn_summary = geography_churn_summary.sort_values(
    by="Churn_Rate",
    ascending=False
)

geography_churn_summary

,Geography,Customer_Count,Churned_Customers,Churn_Rate
1,Germany,2509,814,32.44
2,Spain,2477,413,16.67
0,France,5014,810,16.15


### Gender vs Churn Validation

In [29]:
# Analyze churn rate by gender
gender_churn_summary = (
    bank_df
    .groupby("Gender")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean")
    )
    .reset_index()
)

gender_churn_summary["Churn_Rate"] = (
    gender_churn_summary["Churn_Rate"] * 100
).round(2)

gender_churn_summary = gender_churn_summary.sort_values(
    by="Churn_Rate",
    ascending=False
)

gender_churn_summary

,Gender,Customer_Count,Churned_Customers,Churn_Rate
0,Female,4543,1139,25.07
1,Male,5457,898,16.46


## Data Cleaning & Validation Conclusion

The dataset has been successfully validated for missing values, duplicate records, customer ID uniqueness, binary field consistency, categorical values, numeric ranges, and business logic checks.

Key validation findings:

- The dataset contains 10,000 unique customer records.
- No missing values were found.
- No duplicate records were found.
- Customer IDs are unique.
- Binary fields such as `HasCrCard`, `IsActiveMember`, and `Exited` contain valid values.
- The target variable `Exited` is correctly labeled.
- The dataset contains customer records from only the year 2025.
- Most customers use either 1 or 2 products.
- A significant share of customers have zero account balance.
- Inactive customers show higher churn than active customers.
- Product usage impact is non-linear, with 2-product customers showing stronger retention than 1-product customers, while 3- and 4-product customers show higher churn.
- Credit card ownership does not show a strong churn difference at this stage.
- Germany shows a higher churn rate compared with France and Spain.

Based on the validation checks, the dataset is clean and ready for feature engineering, segmentation, and deeper engagement-retention analysis.

### Export Cleaned Dataset

In [30]:
# Create cleaned data directory if it does not exist
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Export cleaned dataset
bank_df.to_csv(CLEANED_DATA_PATH, index=False)

print(f"Cleaned dataset exported successfully to: {CLEANED_DATA_PATH}")
print(f"Exported Rows: {bank_df.shape[0]:,}")
print(f"Exported Columns: {bank_df.shape[1]:,}")

Cleaned dataset exported successfully to: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\cleaned\european_bank_cleaned.csv
Exported Rows: 10,000
Exported Columns: 14
